# 002 · CSV

> **Standard Library › Data Formats** · Python 3.14

CSV looks trivial until you meet a comma inside a quoted field, an embedded newline, or a semicolon-separated export from a European spreadsheet. The `csv` module handles all of that correctly — splitting lines on `","` yourself does not.

**In this notebook you will learn:**

- reading and writing rows with `csv.reader` / `csv.writer`
- working with header-based records using `DictReader` / `DictWriter`
- why files must be opened with `newline=""`
- quoting styles, delimiters, dialects and the `Sniffer`
- that every value comes back as a `str` — and how to convert types

## .NET mental model

The .NET BCL has **no** general-purpose CSV reader/writer. Your options there are:

| .NET | Python |
|---|---|
| `string.Split(',')` (broken for quoted fields) | `line.split(",")` (equally broken) |
| `Microsoft.VisualBasic.FileIO.TextFieldParser` | `csv.reader` |
| **CsvHelper** (NuGet) `CsvReader.GetRecords<T>()` | `csv.DictReader` (dicts, not typed objects) |
| CsvHelper `CsvWriter.WriteRecords(items)` | `csv.DictWriter.writerows(dicts)` |
| CsvHelper `CsvConfiguration { Delimiter = ";" }` | `delimiter=";"` / a `Dialect` |

So Python is ahead here: CSV support is built in. The one thing it does *not* do is typed mapping — every field is a string.

## Reading with `csv.reader`

`csv.reader` takes any iterable of lines (an open file, a `StringIO`, even a list of strings) and yields each row as a `list[str]`. For demos we use `io.StringIO` *(see Standard Library › Files & Paths › 005 · In-Memory Streams with io)*.

In [ ]:
import csv
import io

data = '''id,name,price
1,apple,0.50
2,"pear, conference",0.75
3,"the ""best"" melon",2.10
'''

for row in csv.reader(io.StringIO(data)):
    print(row)

Notice that the quoted comma and the doubled `""` escape were handled for you — that is exactly what `line.split(",")` gets wrong:

In [ ]:
print('2,"pear, conference",0.75'.split(","))   # 4 pieces — wrong!

## Writing with `csv.writer`

`writerow` takes one sequence, `writerows` takes many. The writer quotes only when needed (`QUOTE_MINIMAL`), writes `None` as an empty field, and converts numbers with `str()`.

In [ ]:
buffer = io.StringIO()
writer = csv.writer(buffer)
writer.writerow(["id", "name", "price"])
writer.writerows([
    [1, "apple", 0.5],
    [2, "pear, conference", 0.75],
    [3, 'say "hi"', None],
])
print(repr(buffer.getvalue()))

Note the `\r\n` line endings: the default `excel` dialect uses CRLF on every platform (as RFC 4180 specifies). Set `lineterminator="\n"` if you need Unix endings.

## Records with `DictReader` and `DictWriter`

`DictReader` uses the first row as the header and yields a `dict` per row — the closest thing to CsvHelper's `GetRecords<T>()`.

```csharp
// C# with CsvHelper (NuGet)
using var reader = new StreamReader(path);
using var csv = new CsvReader(reader, CultureInfo.InvariantCulture);
foreach (var product in csv.GetRecords<Product>())
    Console.WriteLine(product.Name);
```

In [ ]:
for record in csv.DictReader(io.StringIO(data)):
    print(record["id"], record["name"], record["price"])

reader = csv.DictReader(io.StringIO(data))
print(reader.fieldnames)

`DictWriter` needs the column order up front via `fieldnames`. Missing keys are filled with `restval` (default `""`); unexpected keys raise `ValueError` unless you pass `extrasaction="ignore"`.

In [ ]:
orders = [
    {"order_id": 1001, "customer": "alice", "total": 25.0},
    {"order_id": 1002, "customer": "bob"},                       # no total
    {"order_id": 1003, "customer": "carol", "total": 9.5, "note": "gift"},  # extra key
]

buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=["order_id", "customer", "total"],
                        restval="0", extrasaction="ignore", lineterminator="\n")
writer.writeheader()
writer.writerows(orders)
print(buffer.getvalue())

## ⚠️ Gotcha: every value is a string

There is no type inference. `"0.50"` stays a string, and an empty field is `""`, not `None`. Convert explicitly — a small typed record makes the rest of your code much nicer.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal

@dataclass
class Product:
    id: int
    name: str
    price: Decimal

products = [
    Product(int(r["id"]), r["name"], Decimal(r["price"]))
    for r in csv.DictReader(io.StringIO(data))
]
print(products[0])
print(sum(p.price for p in products))

**Tip:** with `quoting=csv.QUOTE_NONNUMERIC`, the *reader* converts every **unquoted** field to `float` (and the writer quotes every non-number). Handy for purely numeric files, but it raises `ValueError` on any unquoted text.

In [ ]:
numeric = '"sensor",1,2.5\n"t2",3,4\n'
for row in csv.reader(io.StringIO(numeric), quoting=csv.QUOTE_NONNUMERIC):
    print(row)

## Files: always open with `newline=""`

The `csv` module does its own line-ending handling, so the file object must **not** translate newlines. The documentation requires `newline=""` for both reading and writing.

- **Writing** without it on Windows turns the writer's `\r\n` into `\r\r\n` — the infamous blank line between every row in Excel.
- **Reading** without it silently rewrites newlines *inside* quoted fields.

The reading problem shows up on every OS:

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "notes.csv"
    path.write_bytes(b'id,note\r\n1,"line one\r\nline two"\r\n')

    with path.open(encoding="utf-8") as f:              # ⚠️ no newline=""
        print(list(csv.reader(f)))
    with path.open(encoding="utf-8", newline="") as f:  # ✅ correct
        print(list(csv.reader(f)))

⚠️ **Gotcha:** also pass `encoding=` explicitly. CSV exported from Excel is often `utf-8-sig` (UTF-8 with a BOM); reading it as plain `utf-8` leaves `"﻿id"` as the first header name.

A complete write-then-read round trip with a real file:

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "products.csv"

    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "name"])
        writer.writerows([[1, "apple"], [2, "pear"]])

    with path.open(encoding="utf-8", newline="") as f:
        print(list(csv.DictReader(f)))

## Quoting styles

| Constant | Writer behaviour |
|---|---|
| `QUOTE_MINIMAL` (default) | quote only fields containing the delimiter, quote char or newline |
| `QUOTE_ALL` | quote every field |
| `QUOTE_NONNUMERIC` | quote every non-number |
| `QUOTE_NONE` | never quote; requires `escapechar` |
| `QUOTE_STRINGS` (3.12+) | quote strings only; `None` becomes an empty unquoted field |
| `QUOTE_NOTNULL` (3.12+) | quote everything except `None` |

In [ ]:
row = ["a", 1, 2.5, None]
for name in ["QUOTE_MINIMAL", "QUOTE_ALL", "QUOTE_NONNUMERIC", "QUOTE_STRINGS", "QUOTE_NOTNULL"]:
    buffer = io.StringIO()
    csv.writer(buffer, quoting=getattr(csv, name), lineterminator="").writerow(row)
    print(f"{name:17} {buffer.getvalue()}")

## Delimiters and dialects

Pass formatting parameters directly (`delimiter`, `quotechar`, `escapechar`, `lineterminator`, `skipinitialspace`, ...) or bundle them into a **dialect**. Built-in dialects are `excel` (default), `excel-tab` and `unix`.

In [ ]:
print(csv.list_dialects())

semicolons = "id;name;price\n1;apple;0,50\n2;pear;0,75\n"   # European-style export
for row in csv.reader(io.StringIO(semicolons), delimiter=";"):
    print(row)

Register your own dialect when the same settings appear in many places — like a shared `CsvConfiguration` in CsvHelper:

In [ ]:
csv.register_dialect("pipes", delimiter="|", quoting=csv.QUOTE_MINIMAL, lineterminator="\n")

buffer = io.StringIO()
csv.writer(buffer, dialect="pipes").writerows([["id", "name"], [1, "a|b"]])
print(buffer.getvalue())

## Detecting the format with `Sniffer`

When you don't know the delimiter in advance, `csv.Sniffer` can guess it from a sample, and guess whether the first row is a header. It is a heuristic — great for "upload any CSV" features, not a substitute for knowing your format.

In [ ]:
sample = "id;name;price\n1;apple;0.50\n2;pear;0.75\n"

sniffer = csv.Sniffer()
dialect = sniffer.sniff(sample)
print(repr(dialect.delimiter))
print(sniffer.has_header(sample))

for row in csv.reader(io.StringIO(sample), dialect):
    print(row)

## Errors

Malformed input raises `csv.Error`. It is rare in the default lenient mode, but `strict=True` makes the parser complain about stray quotes.

In [ ]:
try:
    list(csv.reader(io.StringIO('a,"b"c\n'), strict=True))
except csv.Error as e:
    print(f"{type(e).__name__}: {e}")

**Tip:** for heavy analytical work (millions of rows, type inference, grouping) the usual choice is a third-party library such as pandas or polars. For plain import/export, `csv` is all you need.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Read rows | `TextFieldParser.ReadFields()` | `csv.reader(f)` → `list[str]` |
| Read records | CsvHelper `GetRecords<T>()` | `csv.DictReader(f)` → `dict[str, str]` |
| Write rows | CsvHelper `WriteField` / `NextRecord` | `csv.writer(f).writerow(row)` |
| Write records | CsvHelper `WriteRecords(items)` | `csv.DictWriter(f, fieldnames).writerows(dicts)` |
| Custom delimiter | `CsvConfiguration { Delimiter = ";" }` | `delimiter=";"` or a dialect |
| Open the file | `new StreamReader(path)` | `open(path, newline="", encoding="utf-8")` |
| Typed values | CsvHelper class maps | convert manually (`int(...)`, `Decimal(...)`) |
| Guess the format | — | `csv.Sniffer().sniff(sample)` |

## Exercises

1. Parse `"sku,qty\nA1,3\nB2,5\nA1,2\n"` with `DictReader` and compute the total quantity per SKU (as `int`s).
2. Write the list `[{"city": "Paris, FR", "temp": 21}, {"city": "Oslo", "temp": 15}]` to a tab-separated string with a header, using `DictWriter`.
3. Inside a `TemporaryDirectory`, write three rows to `data.csv` with `csv.writer`, then read them back and check that the round trip preserves a field containing a comma and a newline.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import csv, io, tempfile
from collections import Counter
from pathlib import Path

# 1 — remember to convert qty to int
totals = Counter()
for r in csv.DictReader(io.StringIO("sku,qty\nA1,3\nB2,5\nA1,2\n")):
    totals[r["sku"]] += int(r["qty"])
print(dict(totals))    # {'A1': 5, 'B2': 5}

# 2 — the excel-tab dialect (or delimiter="\t")
buffer = io.StringIO()
writer = csv.DictWriter(buffer, fieldnames=["city", "temp"], dialect="excel-tab")
writer.writeheader()
writer.writerows([{"city": "Paris, FR", "temp": 21}, {"city": "Oslo", "temp": 15}])
print(buffer.getvalue())

# 3 — newline="" on both sides
rows = [["id", "text"], ["1", "plain"], ["2", "comma, and\nnewline"]]
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "data.csv"
    with path.open("w", encoding="utf-8", newline="") as f:
        csv.writer(f).writerows(rows)
    with path.open(encoding="utf-8", newline="") as f:
        print(list(csv.reader(f)) == rows)   # True
```

</details>